# GLiNER2 relation extraction

In this notebook, we demonstrate how to use GLiNER2 for relation extraction tasks, including zero-shot extraction and LoRA training. This has the advantage of being able to extract relations from text without the need for extensive labeled data, while also allowing for fine-tuning on specific datasets to improve performance.

---
## table of contents
- [1. zero-shot](#1.-Zero-shot)
- [2. LoRA](#2.-LoRA)
- [3. fine-tune](#3.-Fine-tune)
---

## 1. Zero-shot

Zero-shot relation extraction allows you to extract relations from text without any prior training on a specific dataset. This is useful for quickly testing the capabilities of GLiNER2 on new data or for extracting relations from text in languages or domains where labeled data is scarce. GLiNER2 can be used to extract relations from text in a zero-shot manner by specifying the relation types you are interested in. The model will then attempt to identify these relations in the provided text based on its pre-trained knowledge.

‼️ Important to note: The quality of the extracted relations may vary depending on the complexity of the text and the specificity of the relation types. It is recommended to evaluate the results and, if necessary, fine-tune the model on a labeled dataset for improved performance.

In [1]:
!pip install gliner2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 28.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import json

In [1]:
## test with Latin text and one relation ----------------
from gliner2 import GLiNER2

extractor = GLiNER2.from_pretrained("GhentCDH/Latin-hagiography-NER")

#fastino/gliner2-multi-v1
#GhentCDH/Latin-hagiography-NER

# Extract relations
text = "Dicitur tamen hæc sancta virgo Glodesinda fuisse temporibus Childerici regis, & cujusdam illustris ducis filia, qui dux Wintro vocabatur. Matris quoque ejus nomen Godila erat;"
results = extractor.extract_relations(
    text,
    ["is_related_to"]
)
print(results)

C:\Users\Heike\PycharmProjects\Hagiographics-relation-extraction\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first
{'relation_extraction': {'is_related_to': [('sancta virgo Glodesinda', 'Childerici regis')]}}


In [2]:
## test using the relation schema 'gliner_schema_hagiographics.json'

import json
from gliner2 import GLiNER2

from gliner_to_labelstudio import (load_gliner_schema_config, create_gliner_schema_from_config_file)

SCHEMA_CONFIG_PATH = "./gliner_schema_hagiographics.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

extractor = GLiNER2.from_pretrained("GhentCDH/Latin-hagiography-NER")

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

text = "Demum vero prædicta Virgo sponte pergens Treviris, atque ad prædictam amitam suam Rotlindam perveniens, ibique una cum ea religiosissime degens, sacram didicit Regulam; ut & seipsam instrueret, ac ceteris sanctimonialibus normam daret. His vero omnibus jureperactis, Mettis rediit; atque in supradicta civitate Mettensi datur ei a parentibus suis locus quidam, quem ipsa elegerat; & ditatum possessionibus sanctæ Virgini relinquunt. At illa inibi mox monasterium construxit, quod Subterius vocatur Monasterium, &, numero [Col. 0204F] centenario sanctimonialium illic agmine collecto, diebus vitæ suæ præfuit atque profuit."


results = extractor.extract(text, schema, include_confidence=True)
print (json.dumps(results, indent=2, ensure_ascii=False))


[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first
{
  "entities": {
    "person": [
      {
        "text": "Virgo",
        "confidence": 0.9752334356307983
      },
      {
        "text": "sanctæ Virgini",
        "confidence": 0.9618059396743774
      },
      {
        "text": "amitam",
        "confidence": 0.8592064380645752
      },
      {
        "text": "Rotlindam",
        "confidence": 0.8460177779197693
      }
    ],
    "group": [
      {
        "text": "sanctimonialium",
        "confidence": 0.9659328460693359
      },
      {
        "text": "sanctimonialibus",
        "confidence": 0.9550672769546509
      }
    ],
    "object": [
      {
        "text": "locus",
        "confidence": 0.6240787506103516
      }
    ],
    "divine_entity": [],
    "place": [
      {
        "text": "Mettis",
        "confidence": 0.9312431216239929
      },
      {
        "text": "Treviris",
        "con

In [4]:
# extracting entities and relations from .txt file

import json
from pathlib import Path
from gliner2 import GLiNER2

from gliner_to_labelstudio import (
    load_gliner_schema_config,
    create_gliner_schema_from_config_file,
)

SCHEMA_CONFIG_PATH = "./gliner_schema_hagiographics.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

extractor = GLiNER2.from_pretrained("GhentCDH/Latin-hagiography-NER")

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

input_txt_path = Path("sample_texts/rule-based-example.txt")

with input_txt_path.open("r", encoding="utf-8") as f:
    text = f.read()

results = extractor.extract(text, schema, include_confidence=True)
print(json.dumps(results, indent=2, ensure_ascii=False))

# save results to GLiNER2-rel/<txt_name>_GLirel.json
output_dir = Path("GLiNER2-rel")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{input_txt_path.stem}_GLirel.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved results to {output_path}")

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first
{
  "entities": {
    "person": [
      {
        "text": "Sanctus Amandulus",
        "confidence": 0.9875401258468628
      },
      {
        "text": "Vir sanctus",
        "confidence": 0.8772599697113037
      }
    ],
    "group": [
      {
        "text": "fratres",
        "confidence": 0.9830965995788574
      },
      {
        "text": "peregrinis",
        "confidence": 0.9410030841827393
      },
      {
        "text": "Multi aegroti",
        "confidence": 0.8381801843643188
      }
    ],
    "object": [
      {
        "text": "arca",
        "confidence": 0.9320533871650696
      }
    ],
    "divine_entity": [],
    "place": [
      {
        "text": "colle",
        "confidence": 0.9955949187278748
      },
      {
        "text": "speluncam",
        "confidence": 0.993839681148529
      },
      {
        "text": "radices silvae",
       

When we run these cells, we can see that GLiNER2 is able to extract some relations using the zero-shot approach. However, it misses a lot of them. This is expected, as the model has not been trained on this specific dataset. To improve the performance, we can fine-tune the model using LoRA training on a labeled dataset of entities and relations in a GLiNER compatible format. This will allow the model to learn from the specific examples in the dataset and improve its ability to extract relations from similar texts.

## 2. LoRA-training

What do you need before starting this LoRA training?
- GLiNER2 installed in your current notebook env
- a labeled dataset of entities and relations in a GLiNER compatible format (use ``ìmport-export.ipynb`` to convert from LS to GLiNER format)
- a schema with definitions for both the entities and relations (see ```gliner_schema_hagiographics.json``` for an example)


In [11]:
# only run this if GLiNER2 is not installed yet
!pip install gliner2

### Step 1: split data into training, validation and test sets
Why this matters:
- Training set teaches the model (80%)
- Validation set helps tune choices (10%)
- Test set gives final unbiased evaluation (10%)


In [2]:
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 31.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.0/461.0 kB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.0/34.0 MB 68.3 MB/s eta 0:00:0000:0100:01


In [3]:
##%%
import json
import os
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────────────────

INPUT_FILE   = "LoRA-training/hagio_REX_training_data_53_gliner.json"
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# ── Load, split and save ──────────────────────────────────────────────────────

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)

os.makedirs("LoRA-training/TTdata_53", exist_ok=True)                                             #change dirname if needed

with open("LoRA-training/TTdata_53/train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)

with open("LoRA-training/TTdata_53/test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2, ensure_ascii=False)

print(f"Total:  {len(data)} examples")
print(f"Train:  {len(train_data)} examples ({(1 - TEST_SIZE)*100:.0f}%)")
print(f"Test:   {len(test_data)} examples ({TEST_SIZE*100:.0f}%)")
print(f"\n📁 Saved to ./LoRA-training/TTdata_53/train.json and ./LoRA-training/TTdata_53/test.json")
print('Now run the next cell to split the test file in half (test-validation)')

Total:  53 examples
Train:  42 examples (80%)
Test:   11 examples (20%)

📁 Saved to ./LoRA-training/TTdata_53/train.json and ./LoRA-training/TTdata_53/test.json


In [4]:
##%%
## split the test set in validation and test

import json
import os
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────────────────

INPUT_FILE   = "LoRA-training/TTdata_53/test.json"  # change to needed path
TEST_SIZE    = 0.5  # 50% of the test set will become validation, 50% will remain test
RANDOM_STATE = 42

# ── Load, split and save ──────────────────────────────────────────────────────

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

val_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)

os.makedirs("LoRA-training/TTdata_53", exist_ok=True)                                             #change dirname if needed

with open("LoRA-training/TTdata_53/val.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2, ensure_ascii=False)

with open("LoRA-training/TTdata_53/test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2, ensure_ascii=False)

print(f"Total:  {len(data)} examples")
print(f"Validation:  {len(val_data)} examples ({(1 - TEST_SIZE)*100:.0f}%)")
print(f"Test:   {len(test_data)} examples ({TEST_SIZE*100:.0f}%)")
print(f"\n📁 Saved to ./LoRA-training/TTdata_53/val.json and ./LoRA-training/TTdata_53/test.json")

Total:  11 examples
Validation:  5 examples (50%)
Test:   6 examples (50%)

📁 Saved to ./LoRA-training/TTdata_53/val.json and ./LoRA-training/TTdata_53/test.json


### Step 2: load training data
Load your training split and convert each sample to InputExample objects used by GLiNER2.

In [5]:
##%%
import json
from gliner2.training.data import InputExample

with open("LoRA-training/TTdata_53/train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

train_data = [
    InputExample(text=item["text"], entities=item["entities"], relations=item["relations"])
    for item in data
]

In [6]:
##%%
import json
from gliner2.training.data import InputExample, Relation

with open("LoRA-training/TTdata_53/train.json", "r", encoding="utf-8") as f:
    data = json.load(f)
with open("LoRA-training/TTdata_53/val.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)


def convert_to_input_example(item: dict) -> InputExample:
    text = item["text"]

    # entities: list of {id, start, end, label, text} -> Dict[label, List[mention_text]]
    id_to_text = {}
    entities_by_type: dict[str, list[str]] = {}
    for ent in item.get("entities", []):
        mention = ent.get("text")
        label = ent.get("label")
        if not mention or not label:
            continue
        entities_by_type.setdefault(label, []).append(mention)
        if ent.get("id"):
            id_to_text[ent["id"]] = mention

    # relations: list of {from_id, to_id, label, direction} -> List[Relation]
    relations = []
    for rel in item.get("relations", []):
        label = rel.get("label")
        head_text = id_to_text.get(rel.get("from_id"))
        tail_text = id_to_text.get(rel.get("to_id"))
        if not label or head_text is None or tail_text is None:
            continue  # skip relations we can't resolve to mention text
        relations.append(Relation(name=label, head=head_text, tail=tail_text))

    return InputExample(text=text, entities=entities_by_type, relations=relations)


train_data = [convert_to_input_example(item) for item in data]
eval_data = [convert_to_input_example(item) for item in eval_data]


### Step 3: Configure LoRA training
Here you define hyperparameters such as epochs, learning rates, batch size, and LoRA rank. For a first run, keep default values and change only output directory and experiment name.

In [7]:
##%%
from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

# LoRA configuration
config = TrainingConfig(
    output_dir="LoRA-training/adapter",  # change this to your desired output directory
    experiment_name="Hagiography_REX",            # change this to your desired experiment name

    # Training parameters
    num_epochs=20,
    batch_size=4,
    gradient_accumulation_steps=1,
    encoder_lr=1e-5,
    task_lr=1e-4,

    # LoRA settings
    use_lora=True,                              # Enable LoRA
    lora_r=4,                                   # Rank (4, 8, 16, 32)
    lora_alpha=8.0,                           # Scaling factor (usually 2*r)
    lora_dropout=0.0,                          # Dropout for LoRA layers
    lora_target_modules=["encoder", "span_rep", "classifier"],           # Apply to all encoder layers (query, key, value, dense)
    save_adapter_only=True,                    # Save only adapter (not full model)

    # Optimization
    eval_strategy="epoch",  # Evaluates and saves at end of each epoch
    eval_steps=500,  # Used when eval_strategy="steps"
    logging_steps=50,
    fp16=True,  # Use mixed precision if GPU available

    early_stopping=True,  # Enable early stopping
    early_stopping_patience=3,  # Stop if no improvement for 3 eval
)

/home/hbekaer/Hagiographics-relation-extraction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 4: Train the LoRA adapter
This launches training using the configuration above.

Expected result: a saved adapter folder in your output directory.

In [8]:
##%%

# Load base model
base_model = GLiNER2.from_pretrained("fastino/gliner2-multi-v1") # fastino/gliner2-multi-v1
# Create trainer
trainer = GLiNER2Trainer(model=base_model, config=config)
# Train adapter
trainer.train(train_data=train_data, eval_data=eval_data) # for early stopping, add eval_data=eval_data

# Adapter automatically saved to the output_dir specified in the cell above 

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first


2026-07-29 16:41:24 - INFO - gliner2.training.trainer - Froze all model parameters for LoRA training
2026-07-29 16:41:24 - INFO - gliner2.training.trainer - LoRA setup complete: 774,148 trainable / 307,872,793 total (0.25%)
Validating records: 100%|██████████| 42/42 [00:00<00:00, 7379.39record/s]
2026-07-29 16:41:24 - INFO - gliner2.training.trainer - Optimizer: LoRA params only = 160, LR=0.0001
/home/hbekaer/Hagiographics-relation-extraction/.venv/lib/python3.12/site-packages/gliner2/training/trainer.py:876: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)
2026-07-29 16:41:24 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-07-29 16:41:24 - INFO - gliner2.training.trainer -   Num examples = 42
2026-07-29 16:41:24 - INFO - gliner2.training.trainer -   Num epochs = 20
2026-07-29 16:41:24 - INFO - gliner2.training.trainer -   Batch size =

{'total_steps': 180,
 'total_epochs': 18,
 'total_time_seconds': 51.15580773353577,
 'samples_per_second': 14.074648254024066,
 'best_metric': 326.621826171875,
 'train_metrics_history': [{'loss': 350.3684664916992,
   'classification_loss': 0.0,
   'structure_loss': 258.10357666015625,
   'count_loss': 9.0634765625,
   'learning_rate': 8.333333333333334e-05,
   'epoch': 4.9,
   'step': 50,
   'samples_seen': 200,
   'throughput': 13.986590719492577},
  {'loss': 283.55758178710937,
   'classification_loss': 0.0,
   'structure_loss': 332.8292541503906,
   'count_loss': 17.8155517578125,
   'learning_rate': 5.555555555555556e-05,
   'epoch': 9.9,
   'step': 100,
   'samples_seen': 400,
   'throughput': 14.075299660463042},
  {'loss': 256.51963012695313,
   'classification_loss': 0.0,
   'structure_loss': 148.19485473632812,
   'count_loss': 10.213958740234375,
   'learning_rate': 2.777777777777778e-05,
   'epoch': 14.9,
   'step': 150,
   'samples_seen': 600,
   'throughput': 14.10877400

##%% md
### Step 5: Evaluate the trained adapter
Compare base model versus adapter on held-out data.

How to interpret metrics:
- Precision: how many predicted entities are correct
- Recall: how many true entities are found
- F1: balance between precision and recall

Always inspect per-entity scores, not only overall score.

In [29]:
!pip install langdetect
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 53.7 MB/s eta 0:00:00 0:00:01


In [9]:
##%%
import json
import pandas as pd
from gliner2 import GLiNER2

# ── Config ────────────────────────────────────────────────────────────────────

BASE_MODEL = "fastino/gliner2-multi-v1"
ADAPTER_PATH = "LoRA-training/adapter/final"          #change to needed path
TEST_FILE = "LoRA-training/TTdata_53/test.json"                          #change to needed path

ENTITY_TYPES = ["person", "group", "divine_entity","place", "institution", "object"] # 'text_title' removed
RELATION_TYPES = ["is_related_to", "goes_to", "comes_from", "resides_at", "owns", "is_owned_by", "located_at", "acts_on", "founds"] # add more relation types as needed

# ── Load test data ────────────────────────────────────────────────────────────

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)


# ── Per-entity metrics helper ─────────────────────────────────────────────────
def normalize_sample(sample: dict):
    """
    Convert a raw sample (entities as a list of span dicts, relations as
    from_id/to_id dicts) into the {type: [mentions]} / {rel_name: {(head, tail)}}
    shapes needed for evaluation.
    """
    id_to_text = {}
    entities_by_type: dict[str, list] = {}
    for ent in sample.get("entities", []):
        mention = ent.get("text")
        label = ent.get("label")
        if not mention or not label:
            continue
        entities_by_type.setdefault(label, []).append(mention)
        if ent.get("id"):
            id_to_text[ent["id"]] = mention

    relations_by_type: dict[str, set] = {}
    for rel in sample.get("relations", []):
        label = rel.get("label")
        head = id_to_text.get(rel.get("from_id"))
        tail = id_to_text.get(rel.get("to_id"))
        if not label or head is None or tail is None:
            continue
        relations_by_type.setdefault(label, set()).add((head.lower().strip(), tail.lower().strip()))

    return entities_by_type, relations_by_type

def compute_per_entity_metrics(pred_entities_dict, true_entities_dict, entity_types):
    metrics = {}
    for label in entity_types:
        true_spans = {s.lower().strip() for s in true_entities_dict.get(label, [])}
        pred_spans = {s.lower().strip() for s in pred_entities_dict.get(label, [])}

        tp = len(pred_spans & true_spans)
        fp = len(pred_spans - true_spans)
        fn = len(true_spans - pred_spans)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) \
            if (precision + recall) > 0 else 0.0

        metrics[label] = {"precision": precision, "recall": recall, "f1": f1,
                          "tp": tp, "fp": fp, "fn": fn}
    return metrics


# ── Relation helpers ──────────────────────────────────────────────────────────

def normalize_true_relations(raw_relations):
    """
    Normalize ground-truth relations from test.json into
    {relation_name: set of (head_text, tail_text)} regardless of whether
    each entry looks like {"name": {"head":..,"tail":..}} (Relation.to_dict()
    style) or {"label"/"name":..,"head":..,"tail":..} (flat style).
    """
    by_type: dict[str, set] = {}
    for rel in raw_relations or []:
        if not isinstance(rel, dict):
            continue

        # Style 1: {"relation_name": {"head": ..., "tail": ...}}
        if len(rel) == 1 and isinstance(next(iter(rel.values())), dict):
            name, fields = next(iter(rel.items()))
            head = fields.get("head")
            tail = fields.get("tail")
        else:
            # Style 2: {"label"/"name": ..., "head": ..., "tail": ...}
            name = rel.get("label") or rel.get("name")
            head = rel.get("head")
            tail = rel.get("tail")

        if not name or head is None or tail is None:
            continue
        by_type.setdefault(name, set()).add((head.lower().strip(), tail.lower().strip()))
    return by_type


def normalize_pred_relations(pred_relation_extraction):
    """
    Normalize model.extract_relations() output ({relation_name: [(head, tail), ...]})
    into {relation_name: set of (head_text, tail_text)}.
    """
    by_type = {}
    for name, pairs in (pred_relation_extraction or {}).items():
        by_type[name] = {(h.lower().strip(), t.lower().strip()) for h, t in pairs if h and t}
    return by_type


def compute_per_relation_metrics(pred_relations_dict, true_relations_dict, relation_types):
    metrics = {}
    for label in relation_types:
        true_pairs = true_relations_dict.get(label, set())
        pred_pairs = pred_relations_dict.get(label, set())

        tp = len(pred_pairs & true_pairs)
        fp = len(pred_pairs - true_pairs)
        fn = len(true_pairs - pred_pairs)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) \
            if (precision + recall) > 0 else 0.0

        metrics[label] = {"precision": precision, "recall": recall, "f1": f1,
                          "tp": tp, "fp": fp, "fn": fn}
    return metrics


# ── Evaluate ──────────────────────────────────────────────────────────────────

model = GLiNER2.from_pretrained(BASE_MODEL)

# Accumulate per-entity and per-relation totals across all samples
all_results = {}
all_relation_results = {}

for run_label, load_adapter in [("Base", False), ("Adapter", True)]:
    if load_adapter:
        model.load_adapter(ADAPTER_PATH)
    else:
        model.unload_adapter()

    # Accumulators: tp, fp, fn per entity/relation type
    accum = {e: {"tp": 0, "fp": 0, "fn": 0} for e in ENTITY_TYPES}
    rel_accum = {r: {"tp": 0, "fp": 0, "fn": 0} for r in RELATION_TYPES}

    for sample in test_data:
        true_entities, true_relations = normalize_sample(sample)

        entity_types = list(true_entities.keys())
        pred = model.extract_entities(sample["text"], entity_types)
        per_entity = compute_per_entity_metrics(
            pred["entities"], true_entities, ENTITY_TYPES
        )
        for label, m in per_entity.items():
            accum[label]["tp"] += m["tp"]
            accum[label]["fp"] += m["fp"]
            accum[label]["fn"] += m["fn"]

        # Relations
        pred_rel_result = model.extract_relations(sample["text"], RELATION_TYPES)
        pred_relations = normalize_pred_relations(pred_rel_result.get("relation_extraction", {}))

        per_relation = compute_per_relation_metrics(
            pred_relations, true_relations, RELATION_TYPES
        )
        for label, m in per_relation.items():
            rel_accum[label]["tp"] += m["tp"]
            rel_accum[label]["fp"] += m["fp"]
            rel_accum[label]["fn"] += m["fn"]
    
    all_results[run_label] = accum
    all_relation_results[run_label] = rel_accum
    print(f"✅ {run_label} evaluated")


# ── Build table ───────────────────────────────────────────────────────────────

def accum_to_metrics(accum_dict, types, label_col="Entity"):
    rows = []
    total_tp = total_fp = total_fn = 0

    for label in types:
        tp = accum_dict[label]["tp"]
        fp = accum_dict[label]["fp"]
        fn = accum_dict[label]["fn"]
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) \
            if (precision + recall) > 0 else 0.0
        rows.append({
            label_col: label,
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1": round(f1, 4),
            "TP": tp,
            "FP": fp,
            "FN": fn,
        })

    # Overall row
    p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    rows.append({
        label_col: "OVERALL",
        "Precision": round(p, 4),
        "Recall": round(r, 4),
        "F1": round(f, 4),
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
    })
    return pd.DataFrame(rows)


df_base = accum_to_metrics(all_results["Base"], ENTITY_TYPES, "Entity")
df_adapter = accum_to_metrics(all_results["Adapter"], ENTITY_TYPES, "Entity")
df = df_base.merge(df_adapter, on="Entity", suffixes=(" (Base)", " (Adapter)"))

df_rel_base = accum_to_metrics(all_relation_results["Base"], RELATION_TYPES, "Relation")
df_rel_adapter = accum_to_metrics(all_relation_results["Adapter"], RELATION_TYPES, "Relation")
df_rel = df_rel_base.merge(df_rel_adapter, on="Relation", suffixes=(" (Base)", " (Adapter)"))


# ── Style and display ─────────────────────────────────────────────────────────

def highlight_rows(row):
    if row.iloc[0] == "OVERALL":
        return ["font-weight: bold; background-color: #f0f0f0"] * len(row)
    return [""] * len(row)


def highlight_better_f1(row):
    styles = [""] * len(row)
    base_f1 = row["F1 (Base)"]
    adapter_f1 = row["F1 (Adapter)"]
    f1_base_idx = row.index.get_loc("F1 (Base)")
    f1_adapter_idx = row.index.get_loc("F1 (Adapter)")
    if adapter_f1 > base_f1:
        styles[f1_adapter_idx] = "color: green; font-weight: bold"
        styles[f1_base_idx] = "color: red"
    elif base_f1 > adapter_f1:
        styles[f1_base_idx] = "color: green; font-weight: bold"
        styles[f1_adapter_idx] = "color: red"
    return styles


fmt = {c: "{:.4f}" for c in df.columns if any(m in c for m in ["Precision", "Recall", "F1"])}
fmt_rel = {c: "{:.4f}" for c in df_rel.columns if any(m in c for m in ["Precision", "Recall", "F1"])}

df.style \
    .apply(highlight_rows, axis=1) \
    .apply(highlight_better_f1, axis=1) \
    .format(fmt) \
    .set_caption("GLiNER2 — Base vs Adapter: per-entity evaluation") \
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "14px"), ("font-weight", "bold")]}])
##%%


2026-07-29 16:43:09 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-multi-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-29 16:43:09 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-multi-v1/cc151f5b0ce4f7010c3ae8884527dd43dddf9d21/config.json "HTTP/1.1 200 OK"
2026-07-29 16:43:09 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-multi-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-29 16:43:09 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-multi-v1/cc151f5b0ce4f7010c3ae8884527dd43dddf9d21/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-07-29 16:43:10 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-multi-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-29 16:43:10 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/

🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first
✅ Base evaluated
✅ Adapter evaluated


,Entity,Precision (Base),Recall (Base),F1 (Base),TP (Base),FP (Base),FN (Base),Precision (Adapter),Recall (Adapter),F1 (Adapter),TP (Adapter),FP (Adapter),FN (Adapter)
0,person,0.5000,0.2727,0.3529,9,9,24,0.6452,0.6061,0.6250,20,11,13
1,group,0.6667,0.4000,0.5000,6,3,9,0.6000,0.6000,0.6000,9,6,6
2,divine_entity,0.5385,0.3889,0.4516,7,6,11,0.7857,0.6111,0.6875,11,3,7
3,place,0.8571,0.2857,0.4286,6,1,15,0.7500,0.7143,0.7317,15,5,6
4,institution,0.6667,0.5000,0.5714,4,2,4,0.5556,0.6250,0.5882,5,4,3
5,object,0.3846,0.6250,0.4762,5,8,3,0.5000,0.7500,0.6000,6,6,2
6,OVERALL,0.5606,0.3592,0.4379,37,29,66,0.6535,0.6408,0.6471,66,35,37


In [10]:
df_rel.style \
    .apply(highlight_rows, axis=1) \
    .apply(highlight_better_f1, axis=1) \
    .format(fmt_rel) \
    .set_caption("GLiNER2 — Base vs Adapter: per-relation evaluation") \
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "14px"), ("font-weight", "bold")]}])

,Relation,Precision (Base),Recall (Base),F1 (Base),TP (Base),FP (Base),FN (Base),Precision (Adapter),Recall (Adapter),F1 (Adapter),TP (Adapter),FP (Adapter),FN (Adapter)
0,is_related_to,1.0000,0.2500,0.4000,1,0,3,0.0000,0.0000,0.0000,0,0,4
1,goes_to,0.0000,0.0000,0.0000,0,1,3,0.0000,0.0000,0.0000,0,1,3
2,comes_from,1.0000,0.5000,0.6667,1,0,1,1.0000,0.5000,0.6667,1,0,1
3,resides_at,0.3333,0.2000,0.2500,1,2,4,0.2500,0.2000,0.2222,1,3,4
4,owns,0.0000,0.0000,0.0000,0,1,4,0.0000,0.0000,0.0000,0,2,4
5,is_owned_by,0.0000,0.0000,0.0000,0,1,0,0.0000,0.0000,0.0000,0,2,0
6,located_at,0.0000,0.0000,0.0000,0,2,0,0.0000,0.0000,0.0000,0,3,0
7,acts_on,0.0000,0.0000,0.0000,0,1,1,0.0000,0.0000,0.0000,0,0,1
8,founds,0.0000,0.0000,0.0000,0,1,0,0.0000,0.0000,0.0000,0,0,0
9,OVERALL,0.2500,0.1579,0.1935,3,9,16,0.1538,0.1053,0.1250,2,11,17


### CONCLUSION:
While a LoRA training requires less data and compute than full fine-tuning, it still requires a large enough dataset to learn the specificities of the relation. In this case, the LoRA adapter trained on 53 texts that contained 263 relations spread over 9 relation types. The evaluation shows no significant improvement over the base model and in some cases even a decrease in performance. This is likely due to the small size of the training dataset, which may not have been sufficient for the model to learn the specificities of the relations. A larger dataset with more examples of each relation type would likely yield better results.

## 3. Fine-tune

Due to the small size of the training dataset and the limited time available to create a larger dataset, we will not be performing a full fine-tuning of the model in this notebook. This step might be performed in a later stage, when more ground truth data is available to us. 